# Yauca/Caravelí 2024 — extracción Vs30: celdas originales auditadas

Derivado sin alterar código ni outputs del notebook `Extraer_Vs30_USGS_Validacion_2024.ipynb` (Drive ID `1sePPNPQnz2g3rC1phL8uKl5m951Bn8Mq`).

Este archivo reproduce **únicamente la extracción de Vs30 para las 44 estaciones** usando `global_vs30.grd`. No debe confundirse con el pipeline completo de Sa, Rrup y predicción V5.1 del evento 2024.


# Extracción de Vs30 USGS para validación temporal 2024

Este notebook monta Google Drive, lee `global_vs30.grd` desde `Tesis/Datos`, extrae Vs30 para las 44 estaciones de validación y guarda el resultado como `Vs30_estaciones_validacion_2024.csv`.

In [ ]:
# Extraer Vs30 USGS para las estaciones de validación temporal 2024
# Ejecutar en Google Colab.
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

GRID = "/content/drive/MyDrive/Tesis/Datos/global_vs30.grd"
OUT  = "/content/drive/MyDrive/Tesis/Datos/Vs30_estaciones_validacion_2024.csv"

stations = pd.DataFrame([{"code": "YCAA", "lat": -15.66, "lon": -74.53}, {"code": "CHAL", "lat": -15.857, "lon": -74.247}, {"code": "ATIC", "lat": -16.224, "lon": -73.616}, {"code": "NASC", "lat": -14.823, "lon": -74.939}, {"code": "CARV", "lat": -15.773, "lon": -73.368}, {"code": "PALP", "lat": -14.538, "lon": -75.187}, {"code": "ANDYA", "lat": -15.797, "lon": -72.863}, {"code": "CCMA", "lat": -16.614, "lon": -72.757}, {"code": "ISABA", "lat": -14.276, "lon": -74.015}, {"code": "COTA", "lat": -15.211, "lon": -72.894}, {"code": "TIBI", "lat": -14.093, "lon": -75.174}, {"code": "UICA", "lat": -14.089, "lon": -75.736}, {"code": "SCICA", "lat": -14.06, "lon": -75.738}, {"code": "PARC", "lat": -14.04, "lon": -75.699}, {"code": "GUA0A", "lat": -13.998, "lon": -75.789}, {"code": "MOLL", "lat": -17.017, "lon": -72.022}, {"code": "PARA", "lat": -13.828, "lon": -76.331}, {"code": "PISCO", "lat": -13.718, "lon": -76.209}, {"code": "PISC", "lat": -13.711, "lon": -76.206}, {"code": "CHIA", "lat": -13.412, "lon": -76.143}, {"code": "SVIC", "lat": -13.075, "lon": -76.387}, {"code": "ILOM", "lat": -17.635, "lon": -71.342}, {"code": "CAZUA", "lat": -12.917, "lon": -76.432}, {"code": "PUCU", "lat": -12.492, "lon": -76.788}, {"code": "SBRT", "lat": -12.39, "lon": -76.774}, {"code": "CITY", "lat": -12.294, "lon": -76.83}, {"code": "LURN", "lat": -12.293, "lon": -76.858}, {"code": "CHRR", "lat": -12.201, "lon": -76.977}, {"code": "CHOR", "lat": -12.179, "lon": -77.008}, {"code": "SAMI", "lat": -12.147, "lon": -76.965}, {"code": "ANRA", "lat": -12.127, "lon": -76.982}, {"code": "RINC", "lat": -12.091, "lon": -76.922}, {"code": "MIRA", "lat": -12.127, "lon": -77.008}, {"code": "CERA", "lat": -12.103, "lon": -76.998}, {"code": "SNIS", "lat": -12.102, "lon": -77.033}, {"code": "BORJA", "lat": -12.086, "lon": -77.006}, {"code": "VICT", "lat": -12.087, "lon": -77.022}, {"code": "SLUI", "lat": -12.073, "lon": -76.999}, {"code": "SANI", "lat": -12.045, "lon": -76.973}, {"code": "MAGD", "lat": -12.095, "lon": -77.072}, {"code": "JEMA", "lat": -12.078, "lon": -77.046}, {"code": "HJUN", "lat": -12.051, "lon": -76.999}, {"code": "TACNA", "lat": -17.954, "lon": -70.184}, {"code": "FEAL", "lat": -11.916, "lon": -77.04}])

# Intento 1: xarray (NetCDF/GMT grid)
try:
    import xarray as xr
    ds = xr.open_dataset(GRID)
    print(ds)

    # Detectar variable de datos y nombres de coordenadas
    data_vars = list(ds.data_vars)
    if not data_vars:
        raise RuntimeError("No se encontró variable de datos en el GRD.")
    vname = data_vars[0]

    lat_candidates = [c for c in ds.coords if c.lower() in ("lat","latitude","y")]
    lon_candidates = [c for c in ds.coords if c.lower() in ("lon","longitude","x")]
    if not lat_candidates or not lon_candidates:
        raise RuntimeError(f"No se detectaron coordenadas lat/lon. Coords: {list(ds.coords)}")
    lat_name, lon_name = lat_candidates[0], lon_candidates[0]

    da = ds[vname]
    vals = []
    for _, r in stations.iterrows():
        val = da.sel({lat_name: float(r.lat), lon_name: float(r.lon)}, method="nearest").item()
        vals.append(float(val))
    stations["Vs30_USGS_m_s"] = vals
    stations["Vs30_fuente"] = "USGS Global Vs30 Mosaic (proxy)"
    stations.to_csv(OUT, index=False)
    print("\nArchivo creado:", OUT)
    print(stations)

except Exception as e:
    print("Falló lectura con xarray:", repr(e))
    print("Intentando con rasterio...")
    import rasterio
    vals = []
    with rasterio.open(GRID) as src:
        for _, r in stations.iterrows():
            val = next(src.sample([(float(r.lon), float(r.lat))]))[0]
            vals.append(float(val))
    stations["Vs30_USGS_m_s"] = vals
    stations["Vs30_fuente"] = "USGS Global Vs30 Mosaic (proxy)"
    stations.to_csv(OUT, index=False)
    print("\nArchivo creado:", OUT)
    print(stations)
